# Linear Programming Optimization**Goal:** Build a real cost matrix and demand profile from the cleaned data, reconcile plant capacity against demand, then formulate and solve a Linear Programming model in PuLP to minimize total distribution cost across the plant-to-customer network.Uses the cleaned dataset produced by `01_data_cleaning.ipynb` (`data/processed/cleaned_orders.csv`).

## 0. Load cleaned data

In [ ]:
import pandas as pdfrom pulp import *final_dedup = pd.read_csv("../data/processed/cleaned_orders.csv")order_list = pd.read_excel("../data/raw/Supply chain logistics problem.xlsx", sheet_name="OrderList", engine="openpyxl")wh_capacities = pd.read_excel("../data/raw/Supply chain logistics problem.xlsx", sheet_name="WhCapacities", engine="openpyxl")print(final_dedup.shape)

## 1. Build the cost matrix and demand profile

In [ ]:
plant_customer_cost = final_dedup.groupby(["Plant Code", "Customer"])["rate"].mean().reset_index()print(plant_customer_cost.head(10))print(plant_customer_cost.shape)

In [ ]:
demand = final_dedup.groupby("Customer")["Unit quantity"].sum().reset_index()print(demand.head(10))print(demand.shape)

## 2. Capacity reconciliationChecking which plants have both cost data (from orders) and capacity data (from `WhCapacities`), and confirming the order data is a single-day snapshot (so no day-multiplication is needed for capacity).

In [ ]:
plants_in_cost = set(plant_customer_cost["Plant Code"].unique())plants_in_capacity =  set(wh_capacities["Plant ID"].unique())print(plants_in_cost & plants_in_capacity)print(plants_in_capacity)print(plants_in_cost)

In [ ]:
print(order_list["Order Date"].min())print(order_list["Order Date"].max())days = (order_list["Order Date"].max() - order_list["Order Date"].min()).daysprint(days)

**Finding:** total plant capacity is ~11,000x smaller than total demand — a clear unit/scale mismatch in the source data, not a real constraint.

In [ ]:
total_demand = demand["Unit quantity"].sum()total_capacity_available = wh_capacities[wh_capacities["Plant ID"].isin(plants_in_cost)]["Daily Capacity "].sum()print("Total demand:", total_demand)print("Total available capacity:", total_capacity_available)

## 3. Proportional capacity scalingRather than inventing arbitrary capacity numbers, each plant's **relative share** of total capacity is preserved and rescaled to the real demand volume, with a 15% buffer for solver flexibility.

In [ ]:
wh_capacities.columns = wh_capacities.columns.str.strip()wh_subset = wh_capacities[wh_capacities["Plant ID"].isin(plants_in_cost)].copy()wh_subset["capacity_share"] = wh_subset["Daily Capacity"] / wh_subset["Daily Capacity"].sum()wh_subset["scaled_capacity"] = wh_subset["capacity_share"] * total_demand * 1.15print(wh_subset)

## 4. Convert to PuLP-friendly dictionaries

In [ ]:
plants = wh_subset["Plant ID"].tolist()customers = demand["Customer"].tolist()print(plants)print(len(customers))capacity = dict(zip(wh_subset["Plant ID"], wh_subset["scaled_capacity"]))print(capacity)

In [ ]:
demand_dict = dict(zip(demand["Customer"], demand["Unit quantity"]))print(list(demand_dict.items())[:5])

In [ ]:
cost_dict = {}for plant in plants:    cost_dict[plant] = {}    for customer in customers:        match = plant_customer_cost[(plant_customer_cost["Plant Code"] == plant) & (plant_customer_cost["Customer"] == customer)]        if not match.empty:            cost_dict[plant][customer] = match["rate"].values[0]print(cost_dict["PLANT03"])

## 5. Build and solve the LP modelA `shortfall` variable (heavily penalized in the objective) is used instead of a hard demand constraint, so the model stays solvable even for customers whose connected plants can't fully meet their demand — rather than returning "Infeasible" with no useful information.

In [ ]:
x = {}for plant in plants:    for customer in customers:        if customer in cost_dict[plant]:            x[(plant, customer)] = LpVariable(f"ship_{plant}_{customer}", lowBound = 0, cat = "Continuous")print(len(x))shortfall = {c : LpVariable(f"shortfall_{c}", lowBound=0) for c in customers}BIG_PENALTY = 1000prob = LpProblem("Plant_Customer_Optimization", LpMinimize)prob += lpSum(cost_dict[p][c] * x[(p,c)] for (p,c) in x) +lpSum(BIG_PENALTY * shortfall[c] for c in customers)for plant in plants:    prob += lpSum(x[(plant, c)] for c in customers if (plant, c) in x) <= capacity[plant]for customer in customers:    prob += lpSum(x[(p, customer)] for p in plants if (p, customer) in x) + shortfall[customer] >= demand_dict[customer]status = prob.solve()print(LpStatus[status])

## 6. Results

In [ ]:
print("total objective value: ", value(prob.objective))actual_shipping_cost = sum(cost_dict[p][c] * x[(p,c)].varValue for (p,c) in x)print("Actual shipping cost:", actual_shipping_cost)total_shortfall = 0for c in customers:    if shortfall[c].varValue > 0:        print(f"{c}: shortfall = {shortfall[c].varValue}")        total_shortfall += shortfall[c].varValueprint("Total unmet demand:", total_shortfall)print("As % of total demand:", (total_shortfall / sum(demand_dict.values())) * 100)

**Result: 97.77% of total demand fulfilled, total logistics cost ₹2.11 crore (~$254K).** A 2.23% shortfall remains across 9 under-connected customers — a genuine network coverage gap in the source data, not a modeling error.

In [ ]:
plant_totals = {}for plant in plants:    total = sum(x[(plant, c)].varValue for c in customers if (plant, c) in x)    plant_totals[plant] = total    print(f"{plant}: shipped {total} out of capacity {capacity[plant]}")

**Sanity check:** 5 of 6 plants are fully utilized (shipped = capacity); PLANT16 retains significant spare capacity — a concrete capacity-rebalancing opportunity.

## 7. Export results

In [ ]:
plant_customer_cost.to_csv("../data/processed/route_cost_matrix.csv", index=False)demand.to_csv("../data/processed/customer_demand.csv", index=False)wh_subset.to_csv("../data/processed/plant_capacity.csv", index=False)print("Saved processed outputs.")